# Data Cleaning Molecule Fingerprints

This Python Notebook is the data Cleaning and data manupulation to Megre the Two data Frames ( Tox21 Molecule) & Validation Monomer data.

- Toxic Molecule Dataset (2202 Molecules) and a 83 Polymer data.

- We will find the cosine Similarty Between both two DFs and then rearrnge them into a single data frame for Further DNN Training input.

# Steps for Process TOXIC Molecule Data Sets and manomer Data set to get Cannonicalised SMILE STRINGS
- Load the Toxic Data sets (/home/sunil/am2/poetry-demo/Polytox_Matser_Thesis/Few_shot_learning_with_validation_data_Monomers/Tox_df_Category.csv)
- read the CSV File
- Drop the unwanted Columns
- Check the Current SMILES are correct or not?
- **Canonicalization** Using "RDKIT" Module
- Add both coulumns in the oroginal data frame ( Canonicalized_smiles, Vaild)
-  Count the False Values ( It should come 32)
-  List the Rows where 'is_valid' is False
- Drop the Rows with "False" Values
- Prepare the **Final Toxin + Manomer  Dataset with "4780" Datapoints with Cananonicalised correct Smile Strings**

In [1]:
import os
import subprocess
import torch

def get_filtered_free_gpus(allowed_gpus={0, 2, 3}):
    """Return (gpu_id, free_mem_MB) from allowed_gpus, sorted by free memory."""
    try:
        result = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=index,memory.free', '--format=csv,noheader,nounits'],
            encoding='utf-8'
        )
        gpu_info = []
        for line in result.strip().split('\n'):
            gpu_id, mem_free = line.strip().split(',')
            gpu_id = int(gpu_id)
            mem_free = int(mem_free)
            if gpu_id in allowed_gpus:
                gpu_info.append((gpu_id, mem_free))
        return sorted(gpu_info, key=lambda x: x[1], reverse=True)
    except Exception as e:
        print("⚠️ Could not query GPU info:", e)
        return []

# Step 1: Get preferred GPU (among 0, 2, 3)
available_gpus = get_filtered_free_gpus()

if available_gpus:
    selected_system_gpu = available_gpus[0][0]
    os.environ["CUDA_VISIBLE_DEVICES"] = str(selected_system_gpu)
    print(f"🧠 Selected system GPU ID: {selected_system_gpu} with {available_gpus[0][1]} MB free memory.")
else:
    print("⚠️ No preferred GPUs available. Using CPU.")

# Step 2: PyTorch setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    current_device = torch.cuda.current_device()
    visible_device_name = torch.cuda.get_device_name(current_device)
    visible_ids = os.environ["CUDA_VISIBLE_DEVICES"].split(",")
    real_system_gpu_id = visible_ids[current_device]

    print("✅ GPU is available!")
    print(f"🖥️ Visible PyTorch Device: cuda:{current_device}")
    print(f"🧭 Actual system GPU ID: {real_system_gpu_id}")
    print(f"📟 GPU Name: {visible_device_name}")
else:
    print("⚠️ GPU not available. Using CPU.")

print(f"Model will run on: {device}")



🧠 Selected system GPU ID: 0 with 40323 MB free memory.
✅ GPU is available!
🖥️ Visible PyTorch Device: cuda:0
🧭 Actual system GPU ID: 0
📟 GPU Name: NVIDIA A100-SXM4-40GB
Model will run on: cuda


In [4]:
import os
import torch
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# 1. Silence parallelism warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# 2. Verify versions
print(f"NumPy version: {np.__version__}")

NumPy version: 1.26.4


In [5]:
!pip install git+https://github.com/kuennethgroup/psmiles.git

import pandas as pd
import numpy as np
from psmiles import PolymerSmiles as PS

Defaulting to user installation because normal site-packages is not writeable
  Cloning https://github.com/kuennethgroup/psmiles.git to /tmp/pip-req-build-3c7jg0ip
  Running command git clone --filter=blob:none --quiet https://github.com/kuennethgroup/psmiles.git /tmp/pip-req-build-3c7jg0ip
  Resolved https://github.com/kuennethgroup/psmiles.git to commit b5b8d3d9e7453feacb0f65aa97266425e2cb699e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
  Using cached pandas-2.3.3-cp310-cp310-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (91 kB)
Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.8 MB)
Using cached pandas-2.3.3-cp310-cp310-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (12.8 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy

In [6]:
# 2. Verify versions
print(f"NumPy version: {np.__version__}")

NumPy version: 1.26.4


In [7]:
from rdkit import Chem
from rdkit.Chem import AllChem
from sentence_transformers import SentenceTransformer      
polyBERT = SentenceTransformer('kuelumbus/polyBERT')

file_path = r"/home/sunil/am2/poetry-demo/AM2_Poly_biodegradebilty/Data_preprocessing_mol/Orgchem_Co2_biodgrade.csv"

mol_df = pd.read_csv(file_path, encoding = "latin1")
mol_df

2026-03-15 21:15:18.015371: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-15 21:15:18.730084: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/home/sunil/.local/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


,Index,chemical_name,smiles,reliability,endpoint,guideline,principle,time(day),biodegradation_percentage
0,1.0,calcium hydrogen borate,[Ca+2].[Ca+2].[Ca+2].[O-]B([O-])[O-].[O-]B([O-...,1.0,Ready,OECD Guideline 301 B,CO2 Evolution,28.0,0.110
1,2.0,1-(diphenylmethyl)piperazine,[Cl-].c1ccc(C(c2ccccc2)[NH+]2CCNCC2)cc1,1.0,Ready,OECD Guideline 301 B,CO2 Evolution,14.0,0.250
2,3.0,Iodo(triphenylphosphino)copper,[Cu+].[I-].c1ccc(P(c2ccccc2)c2ccccc2)cc1,1.0,Ready,OECD Guideline 301 B,CO2 Evolution,28.0,0.120
3,4.0,lithium(1+) ion 12-hydroxyoctadecanoate,[Li]OC(=O)CCCCCCCCCCC(O)CCCCCC,1.0,Ready,OECD Guideline 301 B,CO2 Evolution,28.0,0.970
4,5.0,dilithium(1+) ion 12-hydroxyoctadecanoate octa...,[Li]OC(CCCCCC)CCCCCCCCCCC(=O)OCC(COC(=O)CCCCCC...,1.0,Ready,OECD Guideline 301 B,CO2 Evolution,28.0,0.970
...,...,...,...,...,...,...,...,...,...
2196,2200.0,"2-[(2H-1,3-benzodioxol-5-yl)amino]ethan-1-ol h...",OCCNc1ccc2c(c1)OCO2.[Cl-].[H+],1.0,Ready,OECD Guideline 301 B,CO2 Evolution,21.0,0.085
2197,2201.0,2-[4-(2-hydroxyethoxy)phenoxy]ethan-1-ol,OCCOc1ccc(OCCO)cc1,1.0,Ready,OECD Guideline 301 B,CO2 Evolution,28.0,0.090
2198,2202.0,4-[(morpholin-4-ylsulfanyl)methanethioyl]morph...,S=C(SN1CCOCC1)N1CCOCC1,1.0,Ready,OECD Guideline 301 B,CO2 Evolution,28.0,0.420
2199,2203.0,"O,O,O-triphenyl phosphorothioate",S=P(Oc1ccccc1)(Oc1ccccc1)Oc1ccccc1,1.0,Ready,OECD Guideline 301 B,CO2 Evolution,29.0,0.312


In [32]:
from rdkit import Chem

def process_smiles(smiles):
    # Handle non-string inputs (like NaNs) early
    if not isinstance(smiles, str):
        return None, False
        
    try:
        # Attempt 1: Standard parsing
        mol = Chem.MolFromSmiles(smiles)
        if mol:
            return Chem.MolToSmiles(mol, canonical=True), True
        
        # Attempt 2: More aggressive parsing for "dirty" SMILES
        mol = Chem.MolFromSmiles(smiles, sanitize=False)
        if mol:
            try:
                Chem.SanitizeMol(mol)
                # We return canonical SMILES here to keep the format consistent
                return Chem.MolToSmiles(mol, canonical=True), True
            except:
                pass
        
        return smiles, False
    except Exception:
        return smiles, False

# 1. Drop the columns you mentioned earlier
cols_to_drop = ['Index', 'reliability', 'endpoint', 'guideline', 'principle']
mol_df = mol_df.drop(columns=cols_to_drop, errors='ignore')

# 2. Apply the processing (using the correct column name 'smiles')
# Using .apply() with zip(*) is efficient for returning two columns at once
mol_df['CANONICALISED_SMILES'], mol_df['VALID'] = zip(*mol_df['smiles'].apply(process_smiles))

# 3. Filter out the invalid ones
MOL_df = mol_df[mol_df['VALID'] == True].copy()

MOL_df

[21:47:53] SMILES Parse Error: syntax error while parsing: ydr
[21:47:53] SMILES Parse Error: check for mistakes around position 1:
[21:47:53] ydr
[21:47:53] ^
[21:47:53] SMILES Parse Error: Failed parsing SMILES 'ydr' for input: 'ydr'
[21:47:53] SMILES Parse Error: syntax error while parsing: ydr
[21:47:53] SMILES Parse Error: check for mistakes around position 1:
[21:47:53] ydr
[21:47:53] ^
[21:47:53] SMILES Parse Error: Failed parsing SMILES 'ydr' for input: 'ydr'
[21:47:53] WARNING: not removing hydrogen atom without neighbors
[21:47:53] WARNING: not removing hydrogen atom without neighbors
[21:47:53] WARNING: not removing hydrogen atom without neighbors
[21:47:53] WARNING: not removing hydrogen atom without neighbors
[21:47:53] WARNING: not removing hydrogen atom without neighbors
[21:47:53] WARNING: not removing hydrogen atom without neighbors
[21:47:53] WARNING: not removing hydrogen atom without neighbors
[21:47:53] WARNING: not removing hydrogen atom without neighbors
[21:47:5

,chemical_name,smiles,time(day),biodegradation_percentage,CANONICALISED_SMILES,VALID
0,calcium hydrogen borate,[Ca+2].[Ca+2].[Ca+2].[O-]B([O-])[O-].[O-]B([O-...,28.0,0.110,[Ca+2].[Ca+2].[Ca+2].[O-]B([O-])[O-].[O-]B([O-...,True
1,1-(diphenylmethyl)piperazine,[Cl-].c1ccc(C(c2ccccc2)[NH+]2CCNCC2)cc1,14.0,0.250,[Cl-].c1ccc(C(c2ccccc2)[NH+]2CCNCC2)cc1,True
2,Iodo(triphenylphosphino)copper,[Cu+].[I-].c1ccc(P(c2ccccc2)c2ccccc2)cc1,28.0,0.120,[Cu+].[I-].c1ccc(P(c2ccccc2)c2ccccc2)cc1,True
3,lithium(1+) ion 12-hydroxyoctadecanoate,[Li]OC(=O)CCCCCCCCCCC(O)CCCCCC,28.0,0.970,[Li][O]C(=O)CCCCCCCCCCC(O)CCCCCC,True
4,dilithium(1+) ion 12-hydroxyoctadecanoate octa...,[Li]OC(CCCCCC)CCCCCCCCCCC(=O)OCC(COC(=O)CCCCCC...,28.0,0.970,[Li][O]C(CCCCCC)CCCCCCCCCCC(=O)OCC(COC(=O)CCCC...,True
...,...,...,...,...,...,...
2195,"2-[(2H-1,3-benzodioxol-5-yl)amino]ethan-1-ol h...",OCCNc1ccc2c(c1)OCO2.[Cl-].[H+],14.0,0.085,OCCNc1ccc2c(c1)OCO2.[Cl-].[H+],True
2196,"2-[(2H-1,3-benzodioxol-5-yl)amino]ethan-1-ol h...",OCCNc1ccc2c(c1)OCO2.[Cl-].[H+],21.0,0.085,OCCNc1ccc2c(c1)OCO2.[Cl-].[H+],True
2197,2-[4-(2-hydroxyethoxy)phenoxy]ethan-1-ol,OCCOc1ccc(OCCO)cc1,28.0,0.090,OCCOc1ccc(OCCO)cc1,True
2198,4-[(morpholin-4-ylsulfanyl)methanethioyl]morph...,S=C(SN1CCOCC1)N1CCOCC1,28.0,0.420,S=C(SN1CCOCC1)N1CCOCC1,True


In [33]:
MOL_df

,chemical_name,smiles,time(day),biodegradation_percentage,CANONICALISED_SMILES,VALID
0,calcium hydrogen borate,[Ca+2].[Ca+2].[Ca+2].[O-]B([O-])[O-].[O-]B([O-...,28.0,0.110,[Ca+2].[Ca+2].[Ca+2].[O-]B([O-])[O-].[O-]B([O-...,True
1,1-(diphenylmethyl)piperazine,[Cl-].c1ccc(C(c2ccccc2)[NH+]2CCNCC2)cc1,14.0,0.250,[Cl-].c1ccc(C(c2ccccc2)[NH+]2CCNCC2)cc1,True
2,Iodo(triphenylphosphino)copper,[Cu+].[I-].c1ccc(P(c2ccccc2)c2ccccc2)cc1,28.0,0.120,[Cu+].[I-].c1ccc(P(c2ccccc2)c2ccccc2)cc1,True
3,lithium(1+) ion 12-hydroxyoctadecanoate,[Li]OC(=O)CCCCCCCCCCC(O)CCCCCC,28.0,0.970,[Li][O]C(=O)CCCCCCCCCCC(O)CCCCCC,True
4,dilithium(1+) ion 12-hydroxyoctadecanoate octa...,[Li]OC(CCCCCC)CCCCCCCCCCC(=O)OCC(COC(=O)CCCCCC...,28.0,0.970,[Li][O]C(CCCCCC)CCCCCCCCCCC(=O)OCC(COC(=O)CCCC...,True
...,...,...,...,...,...,...
2195,"2-[(2H-1,3-benzodioxol-5-yl)amino]ethan-1-ol h...",OCCNc1ccc2c(c1)OCO2.[Cl-].[H+],14.0,0.085,OCCNc1ccc2c(c1)OCO2.[Cl-].[H+],True
2196,"2-[(2H-1,3-benzodioxol-5-yl)amino]ethan-1-ol h...",OCCNc1ccc2c(c1)OCO2.[Cl-].[H+],21.0,0.085,OCCNc1ccc2c(c1)OCO2.[Cl-].[H+],True
2197,2-[4-(2-hydroxyethoxy)phenoxy]ethan-1-ol,OCCOc1ccc(OCCO)cc1,28.0,0.090,OCCOc1ccc(OCCO)cc1,True
2198,4-[(morpholin-4-ylsulfanyl)methanethioyl]morph...,S=C(SN1CCOCC1)N1CCOCC1,28.0,0.420,S=C(SN1CCOCC1)N1CCOCC1,True


In [34]:
true_count = MOL_df['VALID'].value_counts().get(True, 0)
print("Number of 'True' in 'VALID':", true_count)

Number of 'True' in 'VALID': 2199


In [35]:
MOL_df.shape

(2199, 6)

In [36]:
# Filter the DataFrame for rows where 'VALID' is False and print them
false_rows_df = MOL_df[MOL_df['VALID'] == False]
print(false_rows_df)

Empty DataFrame
Columns: [chemical_name, smiles, time(day), biodegradation_percentage, CANONICALISED_SMILES, VALID]
Index: []


In [ ]:
# # 1. Find the index of rows where 'VALID' is False
# rows_to_drop = Tox_polymer_df[Tox_polymer_df['VALID'] == False].index

# # 2. Drop these rows by their index
# Tox_polymer_df_cleaned = Tox_polymer_df.drop(rows_to_drop)


In [37]:
MOL_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2199 entries, 0 to 2199
Data columns (total 6 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   chemical_name              2199 non-null   object 
 1   smiles                     2199 non-null   object 
 2   time(day)                  2199 non-null   float64
 3   biodegradation_percentage  2199 non-null   float64
 4   CANONICALISED_SMILES       2199 non-null   object 
 5   VALID                      2199 non-null   bool   
dtypes: bool(1), float64(2), object(3)
memory usage: 105.2+ KB


In [20]:
MOL_df.drop(columns=["VALID", "chemical_name", "smiles"], inplace=True)

In [38]:
MOL_df

,chemical_name,smiles,time(day),biodegradation_percentage,CANONICALISED_SMILES,VALID
0,calcium hydrogen borate,[Ca+2].[Ca+2].[Ca+2].[O-]B([O-])[O-].[O-]B([O-...,28.0,0.110,[Ca+2].[Ca+2].[Ca+2].[O-]B([O-])[O-].[O-]B([O-...,True
1,1-(diphenylmethyl)piperazine,[Cl-].c1ccc(C(c2ccccc2)[NH+]2CCNCC2)cc1,14.0,0.250,[Cl-].c1ccc(C(c2ccccc2)[NH+]2CCNCC2)cc1,True
2,Iodo(triphenylphosphino)copper,[Cu+].[I-].c1ccc(P(c2ccccc2)c2ccccc2)cc1,28.0,0.120,[Cu+].[I-].c1ccc(P(c2ccccc2)c2ccccc2)cc1,True
3,lithium(1+) ion 12-hydroxyoctadecanoate,[Li]OC(=O)CCCCCCCCCCC(O)CCCCCC,28.0,0.970,[Li][O]C(=O)CCCCCCCCCCC(O)CCCCCC,True
4,dilithium(1+) ion 12-hydroxyoctadecanoate octa...,[Li]OC(CCCCCC)CCCCCCCCCCC(=O)OCC(COC(=O)CCCCCC...,28.0,0.970,[Li][O]C(CCCCCC)CCCCCCCCCCC(=O)OCC(COC(=O)CCCC...,True
...,...,...,...,...,...,...
2195,"2-[(2H-1,3-benzodioxol-5-yl)amino]ethan-1-ol h...",OCCNc1ccc2c(c1)OCO2.[Cl-].[H+],14.0,0.085,OCCNc1ccc2c(c1)OCO2.[Cl-].[H+],True
2196,"2-[(2H-1,3-benzodioxol-5-yl)amino]ethan-1-ol h...",OCCNc1ccc2c(c1)OCO2.[Cl-].[H+],21.0,0.085,OCCNc1ccc2c(c1)OCO2.[Cl-].[H+],True
2197,2-[4-(2-hydroxyethoxy)phenoxy]ethan-1-ol,OCCOc1ccc(OCCO)cc1,28.0,0.090,OCCOc1ccc(OCCO)cc1,True
2198,4-[(morpholin-4-ylsulfanyl)methanethioyl]morph...,S=C(SN1CCOCC1)N1CCOCC1,28.0,0.420,S=C(SN1CCOCC1)N1CCOCC1,True


In [39]:
# 1. Identify all current columns
cols = list(MOL_df.columns)

# 2. Remove 'CANONICALISED_SMILES' from its current spot and insert at index 0
cols.insert(0, cols.pop(cols.index('CANONICALISED_SMILES')))

# 3. Reassign the dataframe with the new order
MOL_df = MOL_df[cols]

MOL_df

,CANONICALISED_SMILES,chemical_name,smiles,time(day),biodegradation_percentage,VALID
0,[Ca+2].[Ca+2].[Ca+2].[O-]B([O-])[O-].[O-]B([O-...,calcium hydrogen borate,[Ca+2].[Ca+2].[Ca+2].[O-]B([O-])[O-].[O-]B([O-...,28.0,0.110,True
1,[Cl-].c1ccc(C(c2ccccc2)[NH+]2CCNCC2)cc1,1-(diphenylmethyl)piperazine,[Cl-].c1ccc(C(c2ccccc2)[NH+]2CCNCC2)cc1,14.0,0.250,True
2,[Cu+].[I-].c1ccc(P(c2ccccc2)c2ccccc2)cc1,Iodo(triphenylphosphino)copper,[Cu+].[I-].c1ccc(P(c2ccccc2)c2ccccc2)cc1,28.0,0.120,True
3,[Li][O]C(=O)CCCCCCCCCCC(O)CCCCCC,lithium(1+) ion 12-hydroxyoctadecanoate,[Li]OC(=O)CCCCCCCCCCC(O)CCCCCC,28.0,0.970,True
4,[Li][O]C(CCCCCC)CCCCCCCCCCC(=O)OCC(COC(=O)CCCC...,dilithium(1+) ion 12-hydroxyoctadecanoate octa...,[Li]OC(CCCCCC)CCCCCCCCCCC(=O)OCC(COC(=O)CCCCCC...,28.0,0.970,True
...,...,...,...,...,...,...
2195,OCCNc1ccc2c(c1)OCO2.[Cl-].[H+],"2-[(2H-1,3-benzodioxol-5-yl)amino]ethan-1-ol h...",OCCNc1ccc2c(c1)OCO2.[Cl-].[H+],14.0,0.085,True
2196,OCCNc1ccc2c(c1)OCO2.[Cl-].[H+],"2-[(2H-1,3-benzodioxol-5-yl)amino]ethan-1-ol h...",OCCNc1ccc2c(c1)OCO2.[Cl-].[H+],21.0,0.085,True
2197,OCCOc1ccc(OCCO)cc1,2-[4-(2-hydroxyethoxy)phenoxy]ethan-1-ol,OCCOc1ccc(OCCO)cc1,28.0,0.090,True
2198,S=C(SN1CCOCC1)N1CCOCC1,4-[(morpholin-4-ylsulfanyl)methanethioyl]morph...,S=C(SN1CCOCC1)N1CCOCC1,28.0,0.420,True


In [40]:
MOL_df.shape

(2199, 6)

In [41]:
MOL_df_arr = MOL_df["CANONICALISED_SMILES"].to_numpy()
MOL_df_arr

array(['[Ca+2].[Ca+2].[Ca+2].[O-]B([O-])[O-].[O-]B([O-])[O-]',
       '[Cl-].c1ccc(C(c2ccccc2)[NH+]2CCNCC2)cc1',
       '[Cu+].[I-].c1ccc(P(c2ccccc2)c2ccccc2)cc1', ...,
       'OCCOc1ccc(OCCO)cc1', 'S=C(SN1CCOCC1)N1CCOCC1',
       'S=P(Oc1ccccc1)(Oc1ccccc1)Oc1ccccc1'], dtype=object)

In [42]:
polyBERT = SentenceTransformer('kuelumbus/polyBERT')
MOL_FP= polyBERT.encode(MOL_df_arr)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [43]:
MOL_FP

array([[ 0.7053893 ,  0.6341817 , -0.17331626, ..., -0.58238024,
        -0.66249365,  0.4194019 ],
       [ 0.37490574,  0.5105985 ,  0.24450235, ..., -0.02035966,
         0.1834166 ,  0.5163702 ],
       [ 0.27685788, -0.01824265,  0.48895574, ..., -0.03970774,
         0.1486427 ,  1.1059775 ],
       ...,
       [ 0.18380193,  0.18426593,  0.27035978, ..., -0.58607054,
         0.53408813,  0.9676463 ],
       [-0.5647114 , -0.16697387,  0.2532439 , ..., -0.22670853,
         0.18822227,  0.70241576],
       [-0.02279374,  0.15364008,  0.8964548 , ..., -0.62415344,
         0.4972648 ,  0.94912356]], dtype=float32)

In [44]:
MOL_FP.shape

(2199, 600)

In [45]:
MOL_FP_df =  pd.DataFrame(MOL_FP)
MOL_FP_df

,0,1,2,3,4,5,6,7,8,9,...,590,591,592,593,594,595,596,597,598,599
0,0.705389,0.634182,-0.173316,-0.455370,-0.719308,-0.692111,-0.979560,-0.314649,0.350808,2.377921,...,-0.108624,-1.253606,0.760774,0.775916,0.678984,-1.157291,0.742761,-0.582380,-0.662494,0.419402
1,0.374906,0.510598,0.244502,-0.245051,-0.256825,-0.053042,0.086354,0.234707,0.394395,0.565306,...,0.150706,-0.483368,0.034436,0.443936,0.281356,-0.165448,0.370817,-0.020360,0.183417,0.516370
2,0.276858,-0.018243,0.488956,0.149701,-1.136275,-0.261580,-0.023519,-0.333345,0.166326,0.806764,...,0.073156,-0.368735,-0.474585,0.418816,0.757914,-0.131517,0.547969,-0.039708,0.148643,1.105978
3,0.846700,0.373241,1.214412,-0.269183,-0.932872,-0.265212,0.072404,0.758080,-0.272663,1.552149,...,0.032333,-1.188601,0.189418,-0.798341,0.274286,-0.147163,1.537915,-0.656839,1.105290,-0.066235
4,0.237047,0.003228,1.870494,0.562691,-0.519541,-0.397582,0.517071,0.397954,0.328677,1.295059,...,0.301282,-0.309063,-0.164927,-0.047679,0.202568,0.228564,0.798393,-1.060281,0.368617,-0.430019
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2194,0.338824,0.436833,0.718741,-0.393589,-0.674793,0.154664,0.622563,0.161698,0.047174,1.426566,...,0.209184,-0.405841,0.837518,0.398776,0.405484,-0.749217,0.881655,-0.620740,-0.071931,0.374092
2195,0.338824,0.436833,0.718741,-0.393589,-0.674793,0.154664,0.622563,0.161698,0.047174,1.426566,...,0.209184,-0.405841,0.837518,0.398776,0.405484,-0.749217,0.881655,-0.620740,-0.071931,0.374092
2196,0.183802,0.184266,0.270360,0.587693,-0.740151,-0.077556,1.054792,0.129009,-0.567722,-0.009360,...,0.385736,0.007667,-0.212690,-0.453379,-0.783724,-0.373740,1.101273,-0.586071,0.534088,0.967646
2197,-0.564711,-0.166974,0.253244,-1.578636,-0.311419,0.408021,0.619439,0.528941,-0.262850,0.235161,...,1.143046,-0.113815,0.782955,0.727520,-0.104986,-0.492471,1.100980,-0.226709,0.188222,0.702416


In [46]:
MOL_FP_df.shape

(2199, 600)

In [47]:
MOL_df.shape

(2199, 6)

In [48]:
# Reset the index on both DataFrames
MOL_FP_df.reset_index(drop=True, inplace=True)
MOL_df.reset_index(drop=True, inplace=True)

In [49]:
# Concatenate the DataFrames
MOL_DNN_data = pd.concat([MOL_FP_df, MOL_df], axis=1)

In [50]:
MOL_DNN_data.shape

(2199, 606)

In [51]:
MOL_DNN_data

,0,1,2,3,4,5,6,7,8,9,...,596,597,598,599,CANONICALISED_SMILES,chemical_name,smiles,time(day),biodegradation_percentage,VALID
0,0.705389,0.634182,-0.173316,-0.455370,-0.719308,-0.692111,-0.979560,-0.314649,0.350808,2.377921,...,0.742761,-0.582380,-0.662494,0.419402,[Ca+2].[Ca+2].[Ca+2].[O-]B([O-])[O-].[O-]B([O-...,calcium hydrogen borate,[Ca+2].[Ca+2].[Ca+2].[O-]B([O-])[O-].[O-]B([O-...,28.0,0.110,True
1,0.374906,0.510598,0.244502,-0.245051,-0.256825,-0.053042,0.086354,0.234707,0.394395,0.565306,...,0.370817,-0.020360,0.183417,0.516370,[Cl-].c1ccc(C(c2ccccc2)[NH+]2CCNCC2)cc1,1-(diphenylmethyl)piperazine,[Cl-].c1ccc(C(c2ccccc2)[NH+]2CCNCC2)cc1,14.0,0.250,True
2,0.276858,-0.018243,0.488956,0.149701,-1.136275,-0.261580,-0.023519,-0.333345,0.166326,0.806764,...,0.547969,-0.039708,0.148643,1.105978,[Cu+].[I-].c1ccc(P(c2ccccc2)c2ccccc2)cc1,Iodo(triphenylphosphino)copper,[Cu+].[I-].c1ccc(P(c2ccccc2)c2ccccc2)cc1,28.0,0.120,True
3,0.846700,0.373241,1.214412,-0.269183,-0.932872,-0.265212,0.072404,0.758080,-0.272663,1.552149,...,1.537915,-0.656839,1.105290,-0.066235,[Li][O]C(=O)CCCCCCCCCCC(O)CCCCCC,lithium(1+) ion 12-hydroxyoctadecanoate,[Li]OC(=O)CCCCCCCCCCC(O)CCCCCC,28.0,0.970,True
4,0.237047,0.003228,1.870494,0.562691,-0.519541,-0.397582,0.517071,0.397954,0.328677,1.295059,...,0.798393,-1.060281,0.368617,-0.430019,[Li][O]C(CCCCCC)CCCCCCCCCCC(=O)OCC(COC(=O)CCCC...,dilithium(1+) ion 12-hydroxyoctadecanoate octa...,[Li]OC(CCCCCC)CCCCCCCCCCC(=O)OCC(COC(=O)CCCCCC...,28.0,0.970,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2194,0.338824,0.436833,0.718741,-0.393589,-0.674793,0.154664,0.622563,0.161698,0.047174,1.426566,...,0.881655,-0.620740,-0.071931,0.374092,OCCNc1ccc2c(c1)OCO2.[Cl-].[H+],"2-[(2H-1,3-benzodioxol-5-yl)amino]ethan-1-ol h...",OCCNc1ccc2c(c1)OCO2.[Cl-].[H+],14.0,0.085,True
2195,0.338824,0.436833,0.718741,-0.393589,-0.674793,0.154664,0.622563,0.161698,0.047174,1.426566,...,0.881655,-0.620740,-0.071931,0.374092,OCCNc1ccc2c(c1)OCO2.[Cl-].[H+],"2-[(2H-1,3-benzodioxol-5-yl)amino]ethan-1-ol h...",OCCNc1ccc2c(c1)OCO2.[Cl-].[H+],21.0,0.085,True
2196,0.183802,0.184266,0.270360,0.587693,-0.740151,-0.077556,1.054792,0.129009,-0.567722,-0.009360,...,1.101273,-0.586071,0.534088,0.967646,OCCOc1ccc(OCCO)cc1,2-[4-(2-hydroxyethoxy)phenoxy]ethan-1-ol,OCCOc1ccc(OCCO)cc1,28.0,0.090,True
2197,-0.564711,-0.166974,0.253244,-1.578636,-0.311419,0.408021,0.619439,0.528941,-0.262850,0.235161,...,1.100980,-0.226709,0.188222,0.702416,S=C(SN1CCOCC1)N1CCOCC1,4-[(morpholin-4-ylsulfanyl)methanethioyl]morph...,S=C(SN1CCOCC1)N1CCOCC1,28.0,0.420,True


In [52]:
# 1. Drop the columns you mentioned earlier
cols_to_drop = ['chemical_name', 'smiles', 'VALID']
MOL_DNN_data= MOL_DNN_data.drop(columns=cols_to_drop, errors='ignore')
MOL_DNN_data

,0,1,2,3,4,5,6,7,8,9,...,593,594,595,596,597,598,599,CANONICALISED_SMILES,time(day),biodegradation_percentage
0,0.705389,0.634182,-0.173316,-0.455370,-0.719308,-0.692111,-0.979560,-0.314649,0.350808,2.377921,...,0.775916,0.678984,-1.157291,0.742761,-0.582380,-0.662494,0.419402,[Ca+2].[Ca+2].[Ca+2].[O-]B([O-])[O-].[O-]B([O-...,28.0,0.110
1,0.374906,0.510598,0.244502,-0.245051,-0.256825,-0.053042,0.086354,0.234707,0.394395,0.565306,...,0.443936,0.281356,-0.165448,0.370817,-0.020360,0.183417,0.516370,[Cl-].c1ccc(C(c2ccccc2)[NH+]2CCNCC2)cc1,14.0,0.250
2,0.276858,-0.018243,0.488956,0.149701,-1.136275,-0.261580,-0.023519,-0.333345,0.166326,0.806764,...,0.418816,0.757914,-0.131517,0.547969,-0.039708,0.148643,1.105978,[Cu+].[I-].c1ccc(P(c2ccccc2)c2ccccc2)cc1,28.0,0.120
3,0.846700,0.373241,1.214412,-0.269183,-0.932872,-0.265212,0.072404,0.758080,-0.272663,1.552149,...,-0.798341,0.274286,-0.147163,1.537915,-0.656839,1.105290,-0.066235,[Li][O]C(=O)CCCCCCCCCCC(O)CCCCCC,28.0,0.970
4,0.237047,0.003228,1.870494,0.562691,-0.519541,-0.397582,0.517071,0.397954,0.328677,1.295059,...,-0.047679,0.202568,0.228564,0.798393,-1.060281,0.368617,-0.430019,[Li][O]C(CCCCCC)CCCCCCCCCCC(=O)OCC(COC(=O)CCCC...,28.0,0.970
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2194,0.338824,0.436833,0.718741,-0.393589,-0.674793,0.154664,0.622563,0.161698,0.047174,1.426566,...,0.398776,0.405484,-0.749217,0.881655,-0.620740,-0.071931,0.374092,OCCNc1ccc2c(c1)OCO2.[Cl-].[H+],14.0,0.085
2195,0.338824,0.436833,0.718741,-0.393589,-0.674793,0.154664,0.622563,0.161698,0.047174,1.426566,...,0.398776,0.405484,-0.749217,0.881655,-0.620740,-0.071931,0.374092,OCCNc1ccc2c(c1)OCO2.[Cl-].[H+],21.0,0.085
2196,0.183802,0.184266,0.270360,0.587693,-0.740151,-0.077556,1.054792,0.129009,-0.567722,-0.009360,...,-0.453379,-0.783724,-0.373740,1.101273,-0.586071,0.534088,0.967646,OCCOc1ccc(OCCO)cc1,28.0,0.090
2197,-0.564711,-0.166974,0.253244,-1.578636,-0.311419,0.408021,0.619439,0.528941,-0.262850,0.235161,...,0.727520,-0.104986,-0.492471,1.100980,-0.226709,0.188222,0.702416,S=C(SN1CCOCC1)N1CCOCC1,28.0,0.420


In [53]:
# Get the first 600 columns (fingerprints) as lists
MOL_DNN_data["fingerprints"] = MOL_DNN_data.iloc[:, :600].values.tolist()

MOL_DNN_data = MOL_DNN_data.iloc[:, 600:]

MOL_DNN_data

,CANONICALISED_SMILES,time(day),biodegradation_percentage,fingerprints
0,[Ca+2].[Ca+2].[Ca+2].[O-]B([O-])[O-].[O-]B([O-...,28.0,0.110,"[0.7053893208503723, 0.6341816782951355, -0.17..."
1,[Cl-].c1ccc(C(c2ccccc2)[NH+]2CCNCC2)cc1,14.0,0.250,"[0.3749057352542877, 0.5105984807014465, 0.244..."
2,[Cu+].[I-].c1ccc(P(c2ccccc2)c2ccccc2)cc1,28.0,0.120,"[0.2768578827381134, -0.01824265345931053, 0.4..."
3,[Li][O]C(=O)CCCCCCCCCCC(O)CCCCCC,28.0,0.970,"[0.846700131893158, 0.3732406795024872, 1.2144..."
4,[Li][O]C(CCCCCC)CCCCCCCCCCC(=O)OCC(COC(=O)CCCC...,28.0,0.970,"[0.23704738914966583, 0.0032284678891301155, 1..."
...,...,...,...,...
2194,OCCNc1ccc2c(c1)OCO2.[Cl-].[H+],14.0,0.085,"[0.3388243019580841, 0.43683338165283203, 0.71..."
2195,OCCNc1ccc2c(c1)OCO2.[Cl-].[H+],21.0,0.085,"[0.3388243019580841, 0.43683338165283203, 0.71..."
2196,OCCOc1ccc(OCCO)cc1,28.0,0.090,"[0.18380193412303925, 0.18426592648029327, 0.2..."
2197,S=C(SN1CCOCC1)N1CCOCC1,28.0,0.420,"[-0.5647113919258118, -0.16697387397289276, 0...."


In [54]:
# Step 2: Move fingerprints column to first position
cols = MOL_DNN_data.columns.tolist()
cols.remove('fingerprints')
cols.insert(1, 'fingerprints')
MOL_DNN_data = MOL_DNN_data[cols]
MOL_DNN_data 

,CANONICALISED_SMILES,fingerprints,time(day),biodegradation_percentage
0,[Ca+2].[Ca+2].[Ca+2].[O-]B([O-])[O-].[O-]B([O-...,"[0.7053893208503723, 0.6341816782951355, -0.17...",28.0,0.110
1,[Cl-].c1ccc(C(c2ccccc2)[NH+]2CCNCC2)cc1,"[0.3749057352542877, 0.5105984807014465, 0.244...",14.0,0.250
2,[Cu+].[I-].c1ccc(P(c2ccccc2)c2ccccc2)cc1,"[0.2768578827381134, -0.01824265345931053, 0.4...",28.0,0.120
3,[Li][O]C(=O)CCCCCCCCCCC(O)CCCCCC,"[0.846700131893158, 0.3732406795024872, 1.2144...",28.0,0.970
4,[Li][O]C(CCCCCC)CCCCCCCCCCC(=O)OCC(COC(=O)CCCC...,"[0.23704738914966583, 0.0032284678891301155, 1...",28.0,0.970
...,...,...,...,...
2194,OCCNc1ccc2c(c1)OCO2.[Cl-].[H+],"[0.3388243019580841, 0.43683338165283203, 0.71...",14.0,0.085
2195,OCCNc1ccc2c(c1)OCO2.[Cl-].[H+],"[0.3388243019580841, 0.43683338165283203, 0.71...",21.0,0.085
2196,OCCOc1ccc(OCCO)cc1,"[0.18380193412303925, 0.18426592648029327, 0.2...",28.0,0.090
2197,S=C(SN1CCOCC1)N1CCOCC1,"[-0.5647113919258118, -0.16697387397289276, 0....",28.0,0.420


In [55]:
MOL_LOC_FP = MOL_DNN_data
MOL_LOC_FP 


,CANONICALISED_SMILES,fingerprints,time(day),biodegradation_percentage
0,[Ca+2].[Ca+2].[Ca+2].[O-]B([O-])[O-].[O-]B([O-...,"[0.7053893208503723, 0.6341816782951355, -0.17...",28.0,0.110
1,[Cl-].c1ccc(C(c2ccccc2)[NH+]2CCNCC2)cc1,"[0.3749057352542877, 0.5105984807014465, 0.244...",14.0,0.250
2,[Cu+].[I-].c1ccc(P(c2ccccc2)c2ccccc2)cc1,"[0.2768578827381134, -0.01824265345931053, 0.4...",28.0,0.120
3,[Li][O]C(=O)CCCCCCCCCCC(O)CCCCCC,"[0.846700131893158, 0.3732406795024872, 1.2144...",28.0,0.970
4,[Li][O]C(CCCCCC)CCCCCCCCCCC(=O)OCC(COC(=O)CCCC...,"[0.23704738914966583, 0.0032284678891301155, 1...",28.0,0.970
...,...,...,...,...
2194,OCCNc1ccc2c(c1)OCO2.[Cl-].[H+],"[0.3388243019580841, 0.43683338165283203, 0.71...",14.0,0.085
2195,OCCNc1ccc2c(c1)OCO2.[Cl-].[H+],"[0.3388243019580841, 0.43683338165283203, 0.71...",21.0,0.085
2196,OCCOc1ccc(OCCO)cc1,"[0.18380193412303925, 0.18426592648029327, 0.2...",28.0,0.090
2197,S=C(SN1CCOCC1)N1CCOCC1,"[-0.5647113919258118, -0.16697387397289276, 0....",28.0,0.420


In [56]:
MOL_LOC_FP.tail(120)

,CANONICALISED_SMILES,fingerprints,time(day),biodegradation_percentage
2079,O=C1NC(=O)N(Cc2ccccc2)C(=O)C1c1ccccc1,"[0.05858910083770752, 0.3625769019126892, 0.42...",14.0,0.035
2080,O=C1NC(=O)N(Cc2ccccc2)C(=O)C1c1ccccc1,"[0.05858910083770752, 0.3625769019126892, 0.42...",29.0,0.291
2081,O=c1nc[nH]c2ccccc12,"[0.18329137563705444, 0.9043372869491577, 0.20...",28.0,0.740
2082,O=C1NCC2(CCN(CCc3ccccc3)CC2)O1,"[0.07912006974220276, 0.25277066230773926, -0....",10.0,0.260
2083,O=C1NCC2(CCN(CCc3ccccc3)CC2)O1,"[0.07912006974220276, 0.25277066230773926, -0....",28.0,0.260
...,...,...,...,...
2194,OCCNc1ccc2c(c1)OCO2.[Cl-].[H+],"[0.3388243019580841, 0.43683338165283203, 0.71...",14.0,0.085
2195,OCCNc1ccc2c(c1)OCO2.[Cl-].[H+],"[0.3388243019580841, 0.43683338165283203, 0.71...",21.0,0.085
2196,OCCOc1ccc(OCCO)cc1,"[0.18380193412303925, 0.18426592648029327, 0.2...",28.0,0.090
2197,S=C(SN1CCOCC1)N1CCOCC1,"[-0.5647113919258118, -0.16697387397289276, 0....",28.0,0.420


In [57]:
MOL_LOC_FP.to_csv("Tox_LOC_FP.csv", index=False)


In [58]:
# Safe drop (ignores if columns don't exist)
columns_to_drop = ['CANONICALISED_SMILES']
MOL_LOC_FP.drop(columns=columns_to_drop, inplace=True, errors='ignore')

/tmp/ipykernel_1999440/1374822044.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  MOL_LOC_FP.drop(columns=columns_to_drop, inplace=True, errors='ignore')


In [59]:
MOL_LOC_FP

,fingerprints,time(day),biodegradation_percentage
0,"[0.7053893208503723, 0.6341816782951355, -0.17...",28.0,0.110
1,"[0.3749057352542877, 0.5105984807014465, 0.244...",14.0,0.250
2,"[0.2768578827381134, -0.01824265345931053, 0.4...",28.0,0.120
3,"[0.846700131893158, 0.3732406795024872, 1.2144...",28.0,0.970
4,"[0.23704738914966583, 0.0032284678891301155, 1...",28.0,0.970
...,...,...,...
2194,"[0.3388243019580841, 0.43683338165283203, 0.71...",14.0,0.085
2195,"[0.3388243019580841, 0.43683338165283203, 0.71...",21.0,0.085
2196,"[0.18380193412303925, 0.18426592648029327, 0.2...",28.0,0.090
2197,"[-0.5647113919258118, -0.16697387397289276, 0....",28.0,0.420


In [60]:
# Define the export path
export_path = "/home/sunil/am2/poetry-demo/AM2_Poly_biodegradebilty/Data_preprocessing_mol/MOL_DNN_data.csv"

# Export to CSV
MOL_LOC_FP.to_csv(export_path, index=False)

print(f"Dataset exported successfully!")
print(f"File saved to: {export_path}")
print(f"Dataset shape: {MOL_LOC_FP.shape}")


Dataset exported successfully!
File saved to: /home/sunil/am2/poetry-demo/AM2_Poly_biodegradebilty/Data_preprocessing_mol/MOL_DNN_data.csv
Dataset shape: (2199, 3)


# Correct Shape: (2199, 3) - Clean and manageable
- 
- fingerprints → Time (In days) features
-
 - Clear Target: % of Biodegrdabilty
 - No Missing Data: All encoded properly

 - Ready for Model Training! 🚀
Your dataset is now perfectly prepared for:

Next Steps:
- Extract fingerprint vectors from the fingerprints column (convert from string to numerical array)
- Combine features (fingerprints + property columns)
- Train-test split
- Build DNN model
- Transfer learning implementation

The fingerprints are currently stored as strings/lists, but we need to convert them to numerical arrays and combine with the property columns